# F1 Race Outcome Prediction — Model Training and Evaluation

**Project:** F1 Statistics and Race Prediction Mobile App (pole-position)
**Data:** FastF1 race and qualifying results, 2021–2025

**Workflow**
1. Setup
2. Load data and check quality
3. Clean data
4. Feature engineering (pre-race information only)
5. Train/test split and missing-value handling
6. Evaluation functions
7. Model comparison with time-based cross-validation (2021–2024)
8. Final training (2021–2024) and testing on the 2025 season
9. Selected models: detailed results, feature importance and race predictions
10. Save models for the backend
11. Summary for the report

**How to run:** click **Run All**. Charts, tables and models are saved to the `outputs` and `models` folders automatically.

## 1. Setup

In [ ]:
# Install required packages (safe to run again; skips anything already installed)
%pip install -q pandas numpy scikit-learn matplotlib joblib

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.base import clone
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, log_loss, brier_score_loss,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

RANDOM_STATE = 42

# File locations (relative to this notebook, which is inside the ml folder)
DATA_PATH = "../data/f1_historical_results_2021_2025.csv"
OUTPUT_DIR = "../outputs"
FIGURE_DIR = "../outputs/figures"
MODEL_DIR = "../models"

for folder in [OUTPUT_DIR, FIGURE_DIR, MODEL_DIR]:
    os.makedirs(folder, exist_ok=True)

print("Setup complete.")

## 2. Load Data and Check Quality

In [ ]:
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Seasons:", sorted(df['season'].unique()))
print("Duplicate rows:", df.duplicated().sum())
df.head()

In [ ]:
# Missing values per column
print("Missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Races per season and winners per race (every race should have exactly 1 winner)
races_per_season = df.groupby('season')['round'].nunique()
winners_per_race = df.groupby(['season', 'round'])['winner'].sum()
print("\nRaces per season:")
print(races_per_season.to_string())
print("\nWinners per race:", winners_per_race.value_counts().to_dict())

# Podium label should match finishing positions 1-3
podium_mismatch = (df['podium'] != (df['race_position'] <= 3).astype(int)).sum()
print("Podium label mismatches:", podium_mismatch)

# Grid position 0 means the driver started from the pit lane
print("Pit-lane starts (grid_position = 0):", (df['grid_position'] == 0).sum())

## 3. Clean Data

- Sort races in chronological order so all "before the race" features use earlier races only.
- **Pit-lane starts:** FastF1 records these as `grid_position = 0`. Left unchanged, the model would read 0 as *better than pole position*. They are replaced with the back of the grid (number of drivers in that race).

In [ ]:
# Chronological order (stable sort keeps row order within each race)
df = df.sort_values(['season', 'round'], kind='stable').reset_index(drop=True)

# Pit-lane start (0) -> back of the grid for that race
field_size = df.groupby(['season', 'round'])['driver'].transform('count')
pit_lane = df['grid_position'] == 0
df.loc[pit_lane, 'grid_position'] = field_size[pit_lane]

print("Pit-lane starts replaced:", pit_lane.sum())
print("Remaining grid_position = 0:", (df['grid_position'] == 0).sum())

## 4. Feature Engineering

All features use **only information available before the race starts**. `race_position` and `race_points` of the current race are never used as inputs.

| Feature | Description |
|---|---|
| `grid_position` | Starting position for the race |
| `qualifying_position` | Qualifying result |
| `previous_finish` | Driver's finish in the previous race of the same season |
| `recent_avg_finish` | Driver's average finish over the previous 3 races (same season) |
| `driver_points_before` | Driver's championship points before the race (resets each season) |
| `constructor_points_before` | Team's championship points before the race (resets each season) |
| `no_previous_finish` | 1 if the driver has no previous race this season |
| `no_recent_history` | 1 if the driver has no recent race history this season |

In [ ]:
# Previous finish within the same season
df['previous_finish'] = (
    df.groupby(['season', 'driver'])['race_position']
      .shift(1)
)

# Average finishing position of the previous 3 races (same season)
df['recent_avg_finish'] = (
    df.groupby(['season', 'driver'])['race_position']
      .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

In [ ]:
# Driver championship points before the current race
# FIX: grouped by season AND driver so points reset at the start of every season
# (grouping by driver only carried points over from the previous season)
df['driver_points_before'] = (
    df.groupby(['season', 'driver'])['race_points']
      .transform(lambda x: x.cumsum().shift(1).fillna(0))
)

In [ ]:
# Constructor championship points before the current race
# FIX: team points are totalled per round first, then shifted by one round.
# The old version worked on individual driver rows, so a driver's row could
# include a teammate's points from the SAME race (data leakage).
team_round = (
    df.groupby(['season', 'team', 'round'])['race_points']
      .sum()
      .reset_index()
      .sort_values(['season', 'team', 'round'])
)

team_round['constructor_points_before'] = (
    team_round.groupby(['season', 'team'])['race_points']
      .transform(lambda x: x.cumsum().shift(1).fillna(0))
)

df = df.merge(
    team_round[['season', 'team', 'round', 'constructor_points_before']],
    on=['season', 'team', 'round'],
    how='left'
)

In [ ]:
# Flags for drivers with no race history yet this season
# (created BEFORE missing values are filled, so the information is kept)
df['no_previous_finish'] = df['previous_finish'].isnull().astype(int)
df['no_recent_history'] = df['recent_avg_finish'].isnull().astype(int)

print(df[['no_previous_finish', 'no_recent_history']].sum())

In [ ]:
# Leakage checks
round1 = df[df['round'] == 1]
print("Driver points before round 1:", round1['driver_points_before'].unique())
print("Constructor points before round 1:", round1['constructor_points_before'].unique())

assert (round1['driver_points_before'] == 0).all(), "Driver points leak at round 1"
assert (round1['constructor_points_before'] == 0).all(), "Constructor points leak at round 1"

# Teammates must have the same constructor points before each race
teammate_check = df.groupby(['season', 'round', 'team'])['constructor_points_before'].nunique()
assert (teammate_check == 1).all(), "Teammates have different constructor points"

print("\nAll leakage checks passed.")

# Example: first driver's first races
first_driver = df['driver'].iloc[0]
df[df['driver'] == first_driver][
    ['season', 'round', 'driver_name', 'race_position', 'race_points',
     'driver_points_before', 'constructor_points_before',
     'previous_finish', 'recent_avg_finish']
].head(8)

## 5. Train/Test Split and Missing Values

- **Training:** 2021–2024 seasons. **Testing:** 2025 season (kept unseen until the final evaluation).
- Missing values are filled using **medians from the training data only**, so no information from 2025 is used.

In [ ]:
FEATURES = [
    'grid_position',
    'qualifying_position',
    'driver_points_before',
    'constructor_points_before',
    'previous_finish',
    'recent_avg_finish',
    'no_previous_finish',
    'no_recent_history'
]

IMPUTE_COLUMNS = ['grid_position', 'qualifying_position', 'previous_finish', 'recent_avg_finish']

TARGETS = {'winner': 'Winner', 'podium': 'Podium'}

TEST_SEASON = 2025
train_df = df[df['season'] < TEST_SEASON].copy()
test_df = df[df['season'] == TEST_SEASON].copy()


def fit_imputer(train_part):
    """Median of each column, calculated from training rows only."""
    return train_part[IMPUTE_COLUMNS].median()


def apply_imputer(part, medians):
    part = part.copy()
    part[IMPUTE_COLUMNS] = part[IMPUTE_COLUMNS].fillna(medians)
    return part


train_medians = fit_imputer(train_df)
train_df = apply_imputer(train_df, train_medians)
test_df = apply_imputer(test_df, train_medians)

X_train, X_test = train_df[FEATURES], test_df[FEATURES]

print("Training data:", X_train.shape, "| seasons:", sorted(train_df['season'].unique()))
print("Testing data: ", X_test.shape, "| seasons:", sorted(test_df['season'].unique()))
print("\nMedians used for missing values (from training data):")
print(train_medians.round(2).to_string())
print("\nMissing values after filling:", int(X_train.isnull().sum().sum() + X_test.isnull().sum().sum()))

## 6. Evaluation Functions

Two groups of metrics are used:

- **Classification metrics** (per driver row): accuracy, precision, recall, F1-score, ROC-AUC, log loss and Brier score.
- **Race-level metrics** (match how the app shows predictions):
  - **Top-1 winner hit rate:** in each race, does the driver with the highest predicted probability actually win?
  - **Top-3 podium precision:** of the 3 drivers with the highest podium probability, how many finish on the podium?

In [ ]:
def race_topk(part, scores, target, k):
    """Share of the top-k ranked drivers per race whose actual label is 1."""
    temp = part[['season', 'round', target]].copy()
    temp['score'] = np.asarray(scores)
    temp['rank'] = temp.groupby(['season', 'round'])['score'].rank(method='first', ascending=False)
    top = temp[temp['rank'] <= k]
    return top[target].mean(), int(top[target].sum()), len(top)


def evaluate(part, target, probabilities):
    """All metrics for one model on one dataset."""
    y_true = part[target].values
    y_pred = (probabilities >= 0.5).astype(int)
    k = 1 if target == 'winner' else 3
    topk_rate, correct, total = race_topk(part, probabilities, target, k)
    clipped = np.clip(probabilities, 1e-6, 1 - 1e-6)
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1-score': f1_score(y_true, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_true, probabilities),
        'Log loss': log_loss(y_true, clipped),
        'Brier score': brier_score_loss(y_true, probabilities),
        'Race-level (top-k)': topk_rate,
        'Correct / total': f"{correct} / {total}",
    }


def grid_baseline_scores(part):
    """Simple rule: the better the grid position, the more likely to win/podium."""
    return -part['grid_position'].values


print("Evaluation functions ready.")

## 7. Model Comparison with Time-Based Cross-Validation

Five approaches are compared:

| Model | Why it is included |
|---|---|
| Grid-position baseline | Simple rule (pole sitter wins, top 3 on the grid finish on the podium). Any model should beat this. |
| Logistic Regression | Simple, interpretable linear model |
| Decision Tree | Interpretable, rule-based model |
| Random Forest | Ensemble of many trees (the original model) |
| Gradient Boosting | Ensemble that builds trees step by step to correct earlier errors |

**Time-based (expanding window) cross-validation** uses only past seasons to predict a later season, the same way the app will be used:

| Fold | Train | Validate |
|---|---|---|
| 1 | 2021 | 2022 |
| 2 | 2021–2022 | 2023 |
| 3 | 2021–2023 | 2024 |

The best model for each target is chosen from these validation results. The 2025 test season is **not** used for choosing.

In [ ]:
MODELS = {
    'Logistic Regression': make_pipeline(
        StandardScaler(),
        LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=5, min_samples_leaf=10, class_weight='balanced', random_state=RANDOM_STATE
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, min_samples_leaf=3, class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    'Gradient Boosting': HistGradientBoostingClassifier(
        max_iter=200, learning_rate=0.05, max_depth=4, class_weight='balanced',
        random_state=RANDOM_STATE
    ),
}

raw_dev = df[df['season'] < TEST_SEASON]          # 2021-2024, before filling missing values
dev_seasons = sorted(raw_dev['season'].unique())

cv_rows = []
for val_season in dev_seasons[1:]:
    fold_train = raw_dev[raw_dev['season'] < val_season]
    fold_val = raw_dev[raw_dev['season'] == val_season]

    # Missing values filled with medians from the fold's training seasons only
    medians = fit_imputer(fold_train)
    fold_train = apply_imputer(fold_train, medians)
    fold_val = apply_imputer(fold_val, medians)

    for target in TARGETS:
        # Baseline
        base_scores = grid_baseline_scores(fold_val)
        k = 1 if target == 'winner' else 3
        rate, _, _ = race_topk(fold_val, base_scores, target, k)
        cv_rows.append({'Target': TARGETS[target], 'Model': 'Grid-position baseline',
                        'Validation season': val_season, 'Race-level (top-k)': rate})

        # Machine learning models
        for name, model in MODELS.items():
            fitted = clone(model).fit(fold_train[FEATURES], fold_train[target])
            probs = fitted.predict_proba(fold_val[FEATURES])[:, 1]
            metrics = evaluate(fold_val, target, probs)
            cv_rows.append({'Target': TARGETS[target], 'Model': name,
                            'Validation season': val_season, **metrics})

cv_results = pd.DataFrame(cv_rows)
print("Cross-validation complete:", len(cv_results), "results")

In [ ]:
# Average over the 3 validation seasons
cv_metrics = ['Race-level (top-k)', 'ROC-AUC', 'F1-score', 'Log loss', 'Brier score']
cv_summary = (
    cv_results.groupby(['Target', 'Model'])[cv_metrics]
              .mean()
              .round(3)
              .sort_values(['Target', 'Race-level (top-k)', 'ROC-AUC'], ascending=[True, False, False])
)
cv_summary.to_csv(f"{OUTPUT_DIR}/cv_model_comparison.csv")

# Best model per target: highest race-level score, then highest ROC-AUC as tie-breaker
ml_only = cv_summary.reset_index()
ml_only = ml_only[ml_only['Model'] != 'Grid-position baseline']
best_models = {}
for target, label in TARGETS.items():
    rows = ml_only[ml_only['Target'] == label].sort_values(
        ['Race-level (top-k)', 'ROC-AUC'], ascending=False
    )
    best_models[target] = rows.iloc[0]['Model']

print("Selected models (from cross-validation):")
for target, name in best_models.items():
    print(f"  {TARGETS[target]}: {name}")

print("\nMean cross-validation results (2022-2024 validation seasons):")
cv_summary

In [ ]:
# Chart: race-level performance per model in cross-validation
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
titles = {'Winner': 'Top-1 winner hit rate', 'Podium': 'Top-3 podium precision'}

for ax, (target, label) in zip(axes, TARGETS.items()):
    data = (cv_results[cv_results['Target'] == label]
            .groupby('Model')['Race-level (top-k)'].mean()
            .sort_values())
    colors = ['#9aa5b1' if m == 'Grid-position baseline' else '#2a6fdb' for m in data.index]
    ax.barh(data.index, data.values, color=colors)
    for i, v in enumerate(data.values):
        ax.text(v + 0.01, i, f"{v:.1%}", va='center')
    ax.set_xlim(0, 1)
    ax.set_title(f"{label} model: {titles[label]}")
    ax.set_xlabel('Mean over validation seasons 2022-2024')

plt.tight_layout()
plt.savefig(f"{FIGURE_DIR}/cv_model_comparison.png", dpi=200, bbox_inches='tight')
plt.show()

## 8. Final Training and Testing on 2025

Every model is retrained on all of 2021–2024 and tested once on the unseen 2025 season. This gives the final comparison table for the report.

In [ ]:
final_models = {target: {} for target in TARGETS}
test_probabilities = {target: {} for target in TARGETS}
test_rows = []

for target, label in TARGETS.items():
    # Baseline (race-level only; it gives a ranking, not a probability)
    k = 1 if target == 'winner' else 3
    rate, correct, total = race_topk(test_df, grid_baseline_scores(test_df), target, k)
    test_rows.append({'Target': label, 'Model': 'Grid-position baseline',
                      'Race-level (top-k)': rate, 'Correct / total': f"{correct} / {total}"})

    for name, model in MODELS.items():
        fitted = clone(model).fit(X_train, train_df[target])
        probs = fitted.predict_proba(X_test)[:, 1]
        final_models[target][name] = fitted
        test_probabilities[target][name] = probs
        test_rows.append({'Target': label, 'Model': name, **evaluate(test_df, target, probs)})

test_results = pd.DataFrame(test_rows).set_index(['Target', 'Model'])
test_results.to_csv(f"{OUTPUT_DIR}/test_2025_model_comparison.csv")

pd.options.display.float_format = '{:.3f}'.format
print("2025 test season results (all models):")
test_results

In [ ]:
# Chart: 2025 test results per model
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, (target, label) in zip(axes, TARGETS.items()):
    data = test_results.loc[label, 'Race-level (top-k)'].astype(float).sort_values()
    colors = ['#9aa5b1' if m == 'Grid-position baseline'
              else ('#e8590c' if m == best_models[target] else '#2a6fdb') for m in data.index]
    ax.barh(data.index, data.values, color=colors)
    for i, v in enumerate(data.values):
        ax.text(v + 0.01, i, f"{v:.1%}", va='center')
    ax.set_xlim(0, 1)
    ax.set_title(f"{label} model: {titles[label]} (2025)")
    ax.set_xlabel('Orange = selected model, grey = baseline')

plt.tight_layout()
plt.savefig(f"{FIGURE_DIR}/test_2025_model_comparison.png", dpi=200, bbox_inches='tight')
plt.show()

## 9. Selected Models: Detailed Results

In [ ]:
for target, label in TARGETS.items():
    name = best_models[target]
    probs = test_probabilities[target][name]
    y_pred = (probs >= 0.5).astype(int)
    print(f"===== {label.upper()} MODEL: {name} (2025 test) =====")
    print(classification_report(test_df[target], y_pred, zero_division=0, digits=3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (target, label) in zip(axes, TARGETS.items()):
    name = best_models[target]
    y_pred = (test_probabilities[target][name] >= 0.5).astype(int)
    cm = confusion_matrix(test_df[target], y_pred)
    ConfusionMatrixDisplay(cm, display_labels=[f'Not {label.lower()}', label]).plot(
        ax=ax, cmap='Blues', colorbar=False
    )
    ax.set_title(f"{label} model ({name})")
    print(f"{label} confusion matrix: TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}")

plt.tight_layout()
plt.savefig(f"{FIGURE_DIR}/confusion_matrices.png", dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance (permutation importance on the 2025 test season):
# how much the model's ROC-AUC drops when one feature is randomly shuffled
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
importance_tables = {}

for ax, (target, label) in zip(axes, TARGETS.items()):
    name = best_models[target]
    result = permutation_importance(
        final_models[target][name], X_test, test_df[target],
        scoring='roc_auc', n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1
    )
    imp = pd.Series(result.importances_mean, index=FEATURES).sort_values()
    importance_tables[label] = imp.sort_values(ascending=False).round(4)
    ax.barh(imp.index, imp.values, color='#2a6fdb')
    ax.set_title(f"{label} model ({name})")
    ax.set_xlabel('Drop in ROC-AUC when shuffled')

plt.tight_layout()
plt.savefig(f"{FIGURE_DIR}/feature_importance.png", dpi=200, bbox_inches='tight')
plt.show()

pd.DataFrame(importance_tables)

In [ ]:
# Race-by-race predictions for 2025 (what the app would show)
predictions = test_df[['season', 'round', 'event', 'driver_name', 'team',
                       'grid_position', 'race_position', 'winner', 'podium']].copy()
predictions['winner_probability'] = test_probabilities['winner'][best_models['winner']]
predictions['podium_probability'] = test_probabilities['podium'][best_models['podium']]

# Share within each race (winner chances add up to 100% per race; podium to 300%)
race_groups = predictions.groupby(['season', 'round'])
predictions['win_chance'] = predictions['winner_probability'] / race_groups['winner_probability'].transform('sum')
predictions['podium_chance'] = (3 * predictions['podium_probability']
                                / race_groups['podium_probability'].transform('sum')).clip(upper=1)
predictions['predicted_rank'] = race_groups['winner_probability'].rank(method='first', ascending=False).astype(int)

predictions.sort_values(['round', 'predicted_rank']).to_csv(
    f"{OUTPUT_DIR}/predictions_2025.csv", index=False
)

# Model's pick vs actual winner for every race
picks = predictions[predictions['predicted_rank'] == 1][
    ['round', 'event', 'driver_name', 'win_chance', 'race_position']
].rename(columns={'driver_name': 'predicted_winner', 'race_position': 'pick_finished'})
actual = predictions[predictions['winner'] == 1][['round', 'driver_name']].rename(
    columns={'driver_name': 'actual_winner'}
)
race_summary = picks.merge(actual, on='round', how='left')
race_summary['correct'] = race_summary['predicted_winner'] == race_summary['actual_winner']
race_summary.to_csv(f"{OUTPUT_DIR}/race_by_race_2025.csv", index=False)

print("Correct winner picks:", race_summary['correct'].sum(), "/", len(race_summary))
race_summary

## 10. Save Models for the Backend

Each saved file contains the trained model, the feature list and the training medians, so the FastAPI backend can prepare new inputs in exactly the same way.

The models are retrained on **all seasons (2021–2025)** before saving, so the app uses the most recent data. The evaluation results above remain the official test results.

In [ ]:
full_df = df.copy()
full_medians = fit_imputer(full_df)
full_df = apply_imputer(full_df, full_medians)

for target, label in TARGETS.items():
    name = best_models[target]
    production_model = clone(MODELS[name]).fit(full_df[FEATURES], full_df[target])
    bundle = {
        'model': production_model,
        'model_name': name,
        'target': target,
        'features': FEATURES,
        'impute_medians': full_medians.to_dict(),
        'trained_seasons': sorted(full_df['season'].unique().tolist()),
    }
    path = f"{MODEL_DIR}/{target}_model.joblib"
    joblib.dump(bundle, path)
    print(f"Saved {label} model ({name}) -> {path}")

In [ ]:
# Example: how the backend will load a model and predict one race
def predict_race(race_rows, target='winner'):
    bundle = joblib.load(f"{MODEL_DIR}/{target}_model.joblib")
    X = race_rows[bundle['features']].copy()
    X = X.fillna(bundle['impute_medians'])
    probs = bundle['model'].predict_proba(X)[:, 1]
    out = race_rows[['driver_name', 'team', 'grid_position']].copy()
    out['probability'] = probs
    out['chance'] = probs / probs.sum() if target == 'winner' else (3 * probs / probs.sum()).clip(upper=1)
    return out.sort_values('probability', ascending=False)

last_round = test_df['round'].max()
example_race = df[(df['season'] == TEST_SEASON) & (df['round'] == last_round)]
print("Example prediction:", example_race['event'].iloc[0])
predict_race(example_race, 'winner').head(5)

## 11. Summary for the Report

In [ ]:
print("=" * 60)
print("MODEL TRAINING SUMMARY")
print("=" * 60)
print(f"Dataset: {len(df)} driver-race records, seasons {df['season'].min()}-{df['season'].max()}")
print(f"Training: {len(train_df)} records (2021-2024) | Testing: {len(test_df)} records (2025)")
print(f"Features: {len(FEATURES)}")
print()
for target, label in TARGETS.items():
    name = best_models[target]
    r = test_results.loc[(label, name)]
    b = test_results.loc[(label, 'Grid-position baseline')]
    metric = 'Top-1 winner hit rate' if target == 'winner' else 'Top-3 podium precision'
    print(f"{label} model: {name}")
    print(f"  {metric}: {float(r['Race-level (top-k)']):.1%} ({r['Correct / total']})"
          f"  vs baseline {float(b['Race-level (top-k)']):.1%} ({b['Correct / total']})")
    print(f"  Accuracy {r['Accuracy']:.3f} | Precision {r['Precision']:.3f} | Recall {r['Recall']:.3f}"
          f" | F1 {r['F1-score']:.3f} | ROC-AUC {r['ROC-AUC']:.3f}")
    print()
print("Files saved:")
for folder in [OUTPUT_DIR, FIGURE_DIR, MODEL_DIR]:
    for f in sorted(os.listdir(folder)):
        if os.path.isfile(os.path.join(folder, f)):
            print("  ", os.path.join(folder, f))